# Phase 4 — Agent A1 : Loader / Validator
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre et valide l'agent A1 (`agents/a1_loader.py`) :
- Chargement du CSV de commentaires pré-collectés
- Validation de la colonne `text` (obligatoire)
- Gestion des colonnes optionnelles (`video_id`, `author_likes`, `reply_count`)
- Filtrage des lignes à texte vide

> **Prérequis** : `pip install pandas` — aucun LLM requis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # rend le projet importable

import pandas as pd
from agents.a1_loader import a1_loader, REQUIRED_COLUMNS, OPTIONAL_COLUMNS

print("Colonnes obligatoires :", REQUIRED_COLUMNS)
print("Colonnes optionnelles :", OPTIONAL_COLUMNS)

## 1. Création d'un CSV de test

In [ ]:
import tempfile, pathlib, pandas as pd

# CSV minimal (colonne text obligatoire + optionnelles)
rows = [
    {"text": "This is a great tutorial on machine learning!", "video_id": "abc123", "author_likes": 12, "reply_count": 3},
    {"text": "Very informative, learned a lot about neural networks.", "video_id": "abc123", "author_likes": 8,  "reply_count": 1},
    {"text": "Excellent explanation of gradient descent.", "video_id": "abc123", "author_likes": 5,  "reply_count": 0},
    {"text": "",                                                              "video_id": "abc123", "author_likes": 0,  "reply_count": 0},  # ligne vide → filtrée
    {"text": "Could you cover backpropagation next?",             "video_id": "abc123", "author_likes": 2,  "reply_count": 0},
]

tmp = tempfile.mkdtemp()
csv_path = str(pathlib.Path(tmp) / "comments.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"CSV créé : {csv_path}")
print(pd.read_csv(csv_path))

## 2. Exécution de A1 — cas nominal

In [ ]:
result = a1_loader({"csv_path": csv_path})

print(f"Commentaires chargés : {len(result['raw_comments'])}")
for i, c in enumerate(result["raw_comments"]):
    print(f"  [{i}] text={c['text']!r:50s}  video_id={c.get('video_id')}  likes={c.get('author_likes')}")

## 3. Cas d'erreur

In [ ]:
# Cas 1 : fichier inexistant
err1 = a1_loader({"csv_path": "/nonexistent/path/file.csv"})
print("Cas 1 — fichier manquant:", err1.get("errors"))

# Cas 2 : csv_path vide
err2 = a1_loader({"csv_path": ""})
print("Cas 2 — csv_path vide:", err2.get("errors"))

# Cas 3 : colonne obligatoire manquante
import tempfile, pathlib, pandas as pd
tmp2 = tempfile.mkdtemp()
bad_csv = str(pathlib.Path(tmp2) / "bad.csv")
pd.DataFrame([{"author": "Bob", "likes": 5}]).to_csv(bad_csv, index=False)
err3 = a1_loader({"csv_path": bad_csv})
print("Cas 3 — colonne 'text' absente:", err3.get("errors"))

# Cas 4 : bypass (raw_comments pré-chargé)
bypass = a1_loader({"raw_comments": [{"text": "pre-loaded"}], "csv_path": bad_csv})
print("Cas 4 — bypass raw_comments:", bypass)  # doit retourner {}

## Résumé A1

| Comportement | Résultat |
|---|---|
| CSV valide avec colonnes optionnelles | Tous les records chargés avec métadonnées |
| Ligne texte vide filtrée | Oui |
| CSV manquant | Erreur dans `errors[]` |
| Colonne `text` absente | Erreur dans `errors[]` |
| `raw_comments` pré-chargé | Bypass silencieux (retourne `{}`) |

> **Prochaine étape** : fournir `data/raw/comments.csv` pour exécuter A1 sur données réelles → tag `v0.1.0`